<a href="https://colab.research.google.com/github/karani123-Jeswanth/DSA0302-NLP/blob/main/Program-4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def is_vowel(ch):
    return ch in {'a', 'e', 'i', 'o', 'u'}

def get_vowel_groups(word):
    groups = 0
    in_vowel_group = False
    for ch in word:
        if is_vowel(ch):
            if not in_vowel_group:
                groups += 1
                in_vowel_group = True
        else:
            in_vowel_group = False
    return groups

def is_doublable(stem):
    if len(stem) < 3:
        return False
    c1, v, c2 = stem[-3], stem[-2], stem[-1]
    if is_vowel(c1) or not is_vowel(v) or is_vowel(c2):
        return False
    if c2 in {'w', 'x', 'y'}:
        return False
    if get_vowel_groups(stem) != 1:
        return False
    return True

def apply_rules(stem, feature):
    if feature in {'N+PL', 'V+3SG'}:
        if stem.endswith(('s', 'x', 'z', 'ch', 'sh')):
            return stem + 'es'
        elif len(stem) >= 2 and not is_vowel(stem[-2]) and stem[-1] == 'o':
            return stem + 'es'
        elif len(stem) >= 2 and not is_vowel(stem[-2]) and stem[-1] == 'y':
            return stem[:-1] + 'ies'
        else:
            return stem + 's'

    elif feature == 'V+PROG':
        if len(stem) >= 2 and not is_vowel(stem[-2]) and stem[-1] == 'e':
            return stem[:-1] + 'ing'
        elif is_doublable(stem):
            return stem + stem[-1] + 'ing'
        else:
            return stem + 'ing'

    elif feature in {'V+PAST', 'ADJ+COMP', 'ADJ+SUP'}:
        suffix_map = {
            'V+PAST': ('d', 'ed'),
            'ADJ+COMP': ('r', 'er'),
            'ADJ+SUP': ('st', 'est')
        }
        s1, s2 = suffix_map[feature]
        if stem.endswith('e'):
            return stem + s1
        elif len(stem) >= 2 and not is_vowel(stem[-2]) and stem[-1] == 'y':
            return stem[:-1] + 'i' + s2
        elif is_doublable(stem):
            return stem + stem[-1] + s2
        else:
            return stem + s2

    return stem

def main():
    try:
        s_count = int(input().strip())
        lexicon = []
        for _ in range(s_count):
            line = input().split()
            if len(line) >= 2:
                lexicon.append((line[0], line[1]))

        q_count = int(input().strip())
        queries = []
        for _ in range(q_count):
            line = input().strip()
            if line:
                queries.append(line)

    except (EOFError, ValueError, IndexError):
        return

    surface_to_analyses = {}

    pos_features = {
        'N': ['PL'],
        'V': ['3SG', 'PROG', 'PAST'],
        'ADJ': ['COMP', 'SUP']
    }

    for stem, pos in lexicon:
        bare_analysis = f"{stem}+{pos}"
        surface_to_analyses.setdefault(stem, []).append(bare_analysis)

        if pos in pos_features:
            for feat in pos_features[pos]:
                full_feat = f"{pos}+{feat}"
                surface_form = apply_rules(stem, full_feat)
                analysis_str = f"{stem}+{pos}+{feat}"
                surface_to_analyses.setdefault(surface_form, []).append(analysis_str)

    results = []
    for query in queries:
        if query in surface_to_analyses:
            analyses = sorted(list(set(surface_to_analyses[query])))
            results.append(", ".join(analyses))
        else:
            results.append("UNKNOWN")

    for res in results:
        print(res)

if __name__ == '__main__':
    main()

5
cat N
watch N
watch V
study V
big ADJ
8
cats
watches
watching
studied
bigger
cat
biggest
elephants
cat+N+PL
watch+N+PL, watch+V+3SG
watch+V+PROG
study+V+PAST
big+ADJ+COMP
cat+N
big+ADJ+SUP
UNKNOWN
